# 05 — PPV Validation

Computes positive-predictive-value metrics for the selected tumor-reactive TCRs.

In [ ]:
# Papermill parameter cell
# Default uses scVI outputs; CCA pipeline overrides to data/cca/selected_tumor_reactive_tcrs.csv
selected_tcr_path = 'data/scvi/selected_tumor_reactive_tcrs.csv'
output_ppv_table  = 'data/scvi/ppv_summary.csv'


In [ ]:
import pandas as pd
import numpy as np
import os

print(f'selected_tcr_path : {selected_tcr_path}')
print(f'output_ppv_table  : {output_ppv_table}')


In [ ]:
assert os.path.exists(selected_tcr_path), f'Missing: {selected_tcr_path}'
selected = pd.read_csv(selected_tcr_path)
print(f'Loaded {len(selected)} selected TCRs')
print(selected.head())


In [ ]:
# PPV = (true positives) / (selected as positive)
# Requires an 'is_tumor_reactive' column in the selected TCR table.
# In production, populate this column from a ground-truth reactivity assay
# (e.g. co-culture ELISPOT/ICS results matched to CDR3 sequences).
if 'is_tumor_reactive' in selected.columns:
    n_positive  = selected['is_tumor_reactive'].sum()
    n_selected  = len(selected)
    ppv         = n_positive / n_selected if n_selected > 0 else 0.0
    print(f'Selected TCRs   : {n_selected}')
    print(f'Reactive        : {n_positive}')
    print(f'PPV             : {ppv:.3f}')
    summary = pd.DataFrame([{'n_selected': n_selected, 'n_reactive': int(n_positive), 'ppv': ppv}])
else:
    print('is_tumor_reactive column not present — PPV requires ground-truth labels')
    summary = pd.DataFrame([{'n_selected': len(selected), 'n_reactive': None, 'ppv': None}])
print(summary)


In [ ]:
os.makedirs(os.path.dirname(os.path.abspath(output_ppv_table)), exist_ok=True)
summary.to_csv(output_ppv_table, index=False)
print(f'PPV summary saved -> {output_ppv_table}')
